# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, using the record sets and fields referenced strictly by their `@id` values.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print metadata overview
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset identifier: {metadata.identifier}")
print(f"Published date: {metadata.datePublished}")
print(f"Authors: {[author['@id'] for author in metadata.author]}")


## 2. Data Overview
List available record sets, fields, and their IDs (`@id`).

In [ ]:
# Extract record sets and fields info using @id
record_sets = metadata.recordSet if hasattr(metadata, 'recordSet') and metadata.recordSet else []
if not record_sets:
    print("No record sets found in metadata. Checking 'distribution' for CSV/record set information.")
    distributions = getattr(metadata, 'distribution', [])
    for dist in distributions:
        print(f"Distribution @id: {dist['@id']}")
else:
    for rs in record_sets:
        print(f"Record set @id: {rs['@id']}")
        if hasattr(rs, 'field'):
            fields = rs.field
            if fields:
                for f in fields:
                    print(f"  Field @id: {f['@id']}   Name: {f.get('name', f['@id'])}")

# For this dataset, most record sets are handled via distributions
# We'll extract one distribution and display its meta info
distributions = getattr(metadata, 'distribution', [])
if distributions:
    primary_distribution_id = distributions[0]['@id']
    print(f"\nPrimary distribution @id (record set): {primary_distribution_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s obtained above.

In [ ]:
# For this dataset, we use the distribution @id as the record set.
# We'll extract from all available distributions (record sets).
record_set_ids = [dist['@id'] for dist in metadata.distribution]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"No records loaded for record set @id: {record_set_id}")

# For further processing, select the primary record set
primary_record_set_id = record_set_ids[0]
df_main = dataframes[primary_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply basic processing steps: filtering records, normalizing numeric fields, grouping by demographic fields.
All access to columns is via their `@id` names.

In [ ]:
# Choose numeric field and group field by @id
# We'll try typical survey score columns: GAD-7, PHQ-9, PSQ
# Example column names, as loaded, may be:
# 'gad7_score', 'phq9_score', 'psq_score', etc.
# Instead of using name, we refer to their @id for strict adherence
numeric_field_id = 'gad7_score'  # Replace with the correct @id if available
group_field_id = 'level_of_education'  # Replace with the correct @id

# Check columns
print(f"Available columns: {df_main.columns.tolist()}")
if numeric_field_id in df_main.columns:
    # Filtering: GAD-7 scores above threshold
    threshold = 10
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalizing numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by education level (if available)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not found in columns.")

## 5. Visualization
Visualize distributions and relationships between fields using their `@id`.
We'll plot the distribution of GAD-7 scores and mean scores by education level.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of GAD-7 scores
if numeric_field_id in df_main.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df_main[numeric_field_id], kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id} scores")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Bar plot: mean score by education
    if group_field_id in df_main.columns:
        grouped = df_main.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped)
        plt.title(f"Mean {numeric_field_id} score by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrates how to load, review, extract, and explore the FAIR² Mental Health Survey dataset using `mlcroissant`.

Key findings:
- Data columns are accessible by their `@id` values, supporting robust integration and analysis.
- Survey scores such as GAD-7 can be filtered, normalized, and grouped by demographic attributes (e.g., level_of_education).
- Visualization provides insights into score distributions and demographic effects.

Further analysis may include exploring relationships among other fields, handling missing data, and exporting processed results for ML tasks.